# ACT-JEPA: Illustrated

A compact walkthrough of ACT-JEPA using a tiny model and synthetic inputs. We run the real architecture end to end so the data flow is visible: context encoding, latent future prediction with JEPA, and action decoding.

The model is scaled down for clarity, keeping tensor shapes readable and the notebook quick to run.

## 1. Imports

In [ ]:
import transformers
transformers.logging.set_verbosity_error()

import torch
import torch.nn.functional as F

from models import ActJepaConfig, ActJepaModel

torch.manual_seed(0);

## 2. Configure a Tiny Model

`ActJepaConfig` mirrors the YAML structure under `configs/{env}/act-jepa.yaml`.  
It accepts four sub-configs (`encoder`, `target_encoder`, `predictor`, `decoder`) plus top-level `action_dim`, `state_dim`, and `horizon`.

`horizon` is a dict that tells each sub-module how long each sequence is.  
For example, `observation.state` sets the JEPA prediction horizon, while `action` sets the action chunk size the decoder predicts.

In [ ]:
state_dim   = 4   # size of the robot state vector (e.g. joint positions)
action_dim  = 3   # size of the action vector (e.g. joint velocities)
hidden_size = 64  # transformer embedding dimension

horizon = {"observation.image": 1, "observation.state": 5,
           "episode_index": 1, "task_index": 1, "action": 4}

enc_cfg = dict(hidden_size=hidden_size, intermediate_size=hidden_size*4, num_hidden_layers=1,
               num_attention_heads=4, hidden_act="relu", attention_dropout=0.0,
               horizon=horizon, causal=False)

config = ActJepaConfig(
    action_dim=action_dim, state_dim=state_dim, horizon=horizon,
    encoder=enc_cfg, target_encoder=enc_cfg, predictor=enc_cfg, decoder=enc_cfg,
)

## 3. Instantiate ActJepaModel

`ActJepaModel` consists of 4 components: context encoder $E_\theta$, target encoder $E_{\bar\theta}$, predictor $P_\phi$, and a decoder $D_\tau$.
The target encoder is frozen and updated only via EMA.

In [ ]:
model = ActJepaModel(config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

## 4. Create Synthetic Inputs

The model's `forward(**inputs)` takes in a `dict` of tensors.

| Key | Shape | Notes |
|-----|-------|-------|
| `observation.image` | (B, 3, H, W) | Single RGB frame (ResNet feature extractor) |
| `observation.state` | (B, T_STATE, STATE_DIM) | Full state sequence; context encoder uses only `[:, 0, :]`, target encoder uses all T |
| `observation.state_is_pad` | (B, T_STATE) | `True` = padded step, ignored in JEPA loss |
| `labels` | (B, ACTION_CHUNK, ACTION_DIM) | Ground-truth action chunk |
| `action_is_pad` | (B, ACTION_CHUNK) | `True` = padded action step |
| `task_index` | (B,) | Integer task identifier |

In [ ]:
B = 2

inputs = {
    "observation.image":        torch.randn(B, 3, 64, 64),
    "observation.state":        torch.randn(B, horizon["observation.state"], state_dim),
    "observation.state_is_pad": torch.zeros(B, horizon["observation.state"], dtype=torch.bool),
    "labels":                   torch.randn(B, horizon["action"], action_dim),
    "action_is_pad":            torch.zeros(B, horizon["action"], dtype=torch.bool),
    "task_index":               torch.zeros(B, dtype=torch.long),
}
inputs["observation.state_is_pad"][0, -2:] = True
inputs["action_is_pad"][0, -1:] = True

## 5. Forward Pass

### 5.1 Step by Step

We manually call each component in order to see every intermediate tensor shape, mirroring what `ActJepaModel.forward` does internally.

In [ ]:
model.train()

# Step 1 — Context Encoder: encodes the current state + image + task token
context = model.jepa.context_encoder(**inputs)          # (B, [task, state, and img tokens], C)

# Step 2 — Target Encoder: encodes the full state sequence as JEPA labels (no grad)
model.jepa.target_encoder.eval()  # target encoder stays in eval
with torch.no_grad():
    abstract_labels = model.jepa.target_encoder(**inputs)  # (B, T_STATE, C)

# Step 3 — Predictor: predicts latent future states from context
abstract_pred = model.jepa.predictor(context, **inputs)    # (B, T_STATE, C)

# Step 4 — Decoder + action head: predicts the action chunk from context
action_pred = model.action_head(model.decoder(context, **inputs))  # (B, ACTION_CHUNK, ACTION_DIM)

print(f"context         {tuple(context.shape)}")
print(f"abstract_labels {tuple(abstract_labels.shape)}")
print(f"abstract_pred   {tuple(abstract_pred.shape)}")
print(f"action_pred     {tuple(action_pred.shape)}")

### 5.2 Loss Computation

Two losses are summed to form the final objective:

| Loss | Formula | What it trains |
|------|---------|----------------|
| **Observation loss** $\mathcal{L}_{\text{observations}}$ | $\frac{1}{n}\sum\lVert\hat{s}_{y_{t+i}}-s_{y_{t+i}}\rVert_1$ on non-padded steps | Context encoder + predictor to build a world model |
| **Action loss** $\mathcal{L}_{\text{actions}}$ | $\frac{1}{n}\sum\lVert\hat{a}_{t+i}-a_{t+i}\rVert_1$ on non-padded steps | Context encoder + decoder to predict actions |
| **Total** $\mathcal{L}$ | $\mathcal{L}_{\text{actions}} + \mathcal{L}_{\text{observations}}$ | All trainable components jointly |

In [ ]:
valid_state  = ~inputs["observation.state_is_pad"]
valid_action = ~inputs["action_is_pad"]

obs_loss    = F.l1_loss(abstract_pred[valid_state], abstract_labels[valid_state])
action_loss = F.l1_loss(action_pred[valid_action], inputs["labels"][valid_action])
total_loss  = obs_loss + action_loss

print(f"Observation loss: {obs_loss.item():.4f}")
print(f"Action loss:      {action_loss.item():.4f}")
print(f"Total loss:       {total_loss.item():.4f}")

### 5.3 Full Forward via `model.forward`

In practice you call `model(**inputs)` — it runs all four steps and returns the `output` with both losses ready to backprop.

In [ ]:
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)

model.train()
optimizer.zero_grad()
output = model(**inputs)
output["loss"].backward()
optimizer.step()

print(f"abstract_loss       {output['abstract_loss'].item():.4f}")
print(f"reconstruction_loss {output['reconstruction_loss'].item():.4f}")
print(f"total_loss          {output['loss'].item():.4f}")

## 6. Update of the Target Encoder

After every gradient step the target encoder slowly follows the context encoder:

$$\theta_{\text{target}} \leftarrow m \cdot \theta_{\text{target}} + (1 - m) \cdot \theta_{\text{context}}$$

`model.update_target_encoder(m)` implements this. In the real training loop $m$ ramps from 0.996 → 1.0 (`ema_start` / `ema_end` in the config).

In [ ]:
EMA_M = 0.996

# θ_target ← m * θ_target + (1 - m) * θ_context
model.update_target_encoder(m=EMA_M)

## 7. Mini Training Loop

A 10-step loop wiring everything together: forward → backward → optimizer step → EMA update.

In [ ]:
model = ActJepaModel(config)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)

for step in range(1, 11):
    model.train()
    optimizer.zero_grad()
    out = model(**inputs)
    out["loss"].backward()
    optimizer.step()
    model.update_target_encoder(m=EMA_M)
    print(f"step {step:2d}  abstract_loss={out['abstract_loss'].item():.4f}  action_loss={out['reconstruction_loss'].item():.4f}  total={out['loss'].item():.4f}")